# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the short exercises in [1.7-oop-for-environmental-systems-exercises.ipynb](1.7-oop-for-environmental-systems-exercises.ipynb).
:::

## Exercise 1: A class with state and behaviour

Define a `Glacier` class with `name` and `area_km2` instance attributes, a `describe()` method returning a short string, and a `__repr__`. Create one and print both the object and its description.

In [ ]:
class Glacier:
    def __init__(self, name: str, area_km2: float):
        self.name = name
        self.area_km2 = area_km2

    def describe(self) -> str:
        return f"{self.name}: {self.area_km2} km^2"

    def __repr__(self) -> str:
        return f"Glacier({self.name!r}, {self.area_km2})"

g = Glacier("Aletsch", 78.0)
print(g)
print(g.describe())

## Exercise 2: Instance versus class attributes

Define a `Planet` class with a shared class attribute `g_earth = 9.81` and per-instance attributes `name` and `surface_gravity`. Create two planets and show that `g_earth` is shared while `surface_gravity` differs.

In [ ]:
class Planet:
    g_earth = 9.81                       # shared class attribute

    def __init__(self, name: str, surface_gravity: float):
        self.name = name
        self.surface_gravity = surface_gravity

earth = Planet("Earth", 9.81)
mars = Planet("Mars", 3.71)
print(Planet.g_earth, earth.g_earth, mars.g_earth)         # all 9.81
print(earth.surface_gravity, mars.surface_gravity)         # 9.81, 3.71

## Exercise 3: A dataclass record

Use `@dataclass` to define a `Reading` with `timestamp` (str) and `temp_celsius` (float). Show that two readings with equal fields compare equal, and print one to see the generated repr.

In [ ]:
from dataclasses import dataclass

@dataclass
class Reading:
    timestamp: str
    temp_celsius: float

print(Reading("2024-06-01", 18.2))
print(Reading("2024-06-01", 18.2) == Reading("2024-06-01", 18.2))

## Exercise 4: Read and mutate methods, with validation

Define a `Series` class holding a list of floats, with `add(x)` to append and `mean()` to return the average. `mean()` should raise `ValueError` if there are no values. Demonstrate both.

In [ ]:
class Series:
    def __init__(self):
        self.values: list[float] = []

    def add(self, x: float) -> None:
        self.values.append(x)

    def mean(self) -> float:
        if not self.values:
            raise ValueError("no values")
        return sum(self.values) / len(self.values)

s = Series()
s.add(1.0); s.add(3.0)
print(s.mean())

## Exercise 5: Composition

Define a `Catchment` class that holds several `Series` objects keyed by name (composition). Add an `add_series(name, series)` method and a `names()` method returning the keys. Build one with two series.

In [ ]:
class Catchment:
    def __init__(self, name: str):
        self.name = name
        self.series: dict[str, Series] = {}

    def add_series(self, name: str, series: "Series") -> None:
        self.series[name] = series

    def names(self) -> list[str]:
        return list(self.series)

c = Catchment("Aare")
c.add_series("temp", Series())
c.add_series("discharge", Series())
print(c.names())

## Exercise 6: Fix the shared class-level list

The class below shares one list across all instances. Rewrite it so each instance has its own `entries`, then show two loggers do not contaminate each other.

```python
class Logger:
    entries = []
    def __init__(self, name):
        self.name = name
    def log(self, msg):
        self.entries.append(msg)
```

In [ ]:
class Logger:
    def __init__(self, name):
        self.name = name
        self.entries = []          # per-instance list

    def log(self, msg):
        self.entries.append(msg)

a = Logger("a"); b = Logger("b")
a.log("x"); b.log("y")
print(a.entries, b.entries)        # ['x'] ['y']

## Exercise 7: Inheritance and overriding

Define a base `Station` with a `name` attribute and a `kind()` method returning `"station"`. Define `RiverGauge(Station)` that overrides `kind()` to return `"river gauge"`. Show that a `RiverGauge` keeps the inherited name but reports the new kind.

In [ ]:
class Station:
    def __init__(self, name: str):
        self.name = name

    def kind(self) -> str:
        return "station"

class RiverGauge(Station):
    def kind(self) -> str:           # override
        return "river gauge"

g = RiverGauge("Aare")
print(g.name, g.kind())

## Exercise 8: An invariant with assert

Write `normalise(weights)` that divides each weight by the total, and asserts the invariant that the result sums to 1 (within a small tolerance). Test it on `[2.0, 2.0]`.

In [ ]:
def normalise(weights):
    total = sum(weights)
    out = [w / total for w in weights]
    assert abs(sum(out) - 1.0) < 1e-9, "weights must sum to 1"
    return out

print(normalise([2.0, 2.0]))   # [0.5, 0.5]

## Exercise 9: A custom exception

Define `NegativeDischargeError(ValueError)` and a function `check(q_m3s)` that raises it when discharge is negative. Catch it and print the message.

In [ ]:
class NegativeDischargeError(ValueError):
    pass

def check(q_m3s):
    if q_m3s < 0.0:
        raise NegativeDischargeError(f"discharge {q_m3s} < 0")
    return q_m3s

try:
    check(-3.0)
except NegativeDischargeError as err:
    print("caught:", err)

## Exercise 10: Try / except / else / finally

Write `safe_divide(a, b)` that returns `a / b`, catches `ZeroDivisionError` (returning `None`), prints a message in the `else` branch when it succeeds, and always prints "done" in `finally`. Call it with `(6, 2)` and `(6, 0)`.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
        return None
    else:
        print("division ok")
        return result
    finally:
        print("done")

print(safe_divide(6, 2))
print(safe_divide(6, 0))

## Exercise 11: Validate preconditions

Write `validate(temp_celsius, rh_percent)` that raises `ValueError` if the temperature is below absolute zero or the relative humidity is outside 0–100 %. Show it passing on valid input and raising on `rh_percent = 150`.

In [ ]:
def validate(temp_celsius, rh_percent):
    if temp_celsius < -273.15:
        raise ValueError("temperature below absolute zero")
    if not (0.0 <= rh_percent <= 100.0):
        raise ValueError("relative humidity out of range")

validate(20.0, 55.0)
try:
    validate(20.0, 150.0)
except ValueError as err:
    print("caught:", err)

## Exercise 12: Logging with levels

Configure logging to stdout at INFO level, get a logger, and emit a debug message (which should be suppressed), an info message, and a warning.

In [ ]:
import logging
import sys
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("demo")
log.debug("suppressed")
log.info("started")
log.warning("low sample count")

## Exercise 13: Write and run a pytest suite

Write a module `mymod.py` with `double(x)` returning `2 * x`, and a parametrised test file that checks three cases. Run pytest on it with a subprocess and print the output.

In [ ]:
from pathlib import Path
import subprocess
import sys
Path("_files").mkdir(exist_ok=True)
Path("_files/mymod.py").write_text("def double(x):\n    return 2 * x\n", encoding="utf-8")
Path("_files/test_mymod.py").write_text(
    "import pytest\n"
    "from mymod import double\n"
    "@pytest.mark.parametrize('x, y', [(1, 2), (3, 6), (0, 0)])\n"
    "def test_double(x, y):\n"
    "    assert double(x) == y\n",
    encoding="utf-8",
)
result = subprocess.run([sys.executable, "-m", "pytest", "test_mymod.py", "-q"],
                        capture_output=True, text=True, cwd="_files")
print(result.stdout.strip())

## Exercise 14: Replace assert-based validation

The line `assert rh_percent >= 0, "negative humidity"` disappears under `python -O`. Rewrite the check as a function that raises `ValueError`, so it fires regardless of optimisation. Demonstrate on a valid value.

In [ ]:
def validate_humidity(rh_percent):
    if rh_percent < 0.0:
        raise ValueError("negative humidity")
    return rh_percent

print(validate_humidity(55.0))